In [1]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [2]:
!wget -O prepare_csv.zip https://www.dropbox.com/scl/fi/64ofbvu8hfe3b8k3jc194/prepare_csv.zip?rlkey=fcw86tx0a42sjo7ykrg4l40xw&st=ei35rei1&dl=0
!unzip prepare_csv.zip

Archive:  prepare_csv.zip
replace metadata.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [3]:
# Load csv data
train_features = pd.read_csv('train_features.csv')
train_labels = pd.read_csv('train_labels.csv')

# Average features and combine with labels
feature_columns = train_features.columns[3:]
aggregated_features = train_features.groupby('uid')[feature_columns].mean().reset_index()
merged_df = pd.merge(aggregated_features, train_labels, on='uid')

# Separate features and labels
X = merged_df[feature_columns]
y = merged_df[['diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd']]

# Split the data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Convert labels to numbers
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train.idxmax(axis=1))
y_val_encoded = le.transform(y_val.idxmax(axis=1))

In [4]:
# Train XGBoost classifier
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    random_state=42,
)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_encoded), y=y_train_encoded)

# Create sample weights array
sample_weights = np.ones(len(y_train_encoded))
for idx, label in enumerate(y_train_encoded):
  sample_weights[idx] = class_weights[label]

xgb.fit(X_train_scaled, y_train_encoded, sample_weight=sample_weights)
y_pred = xgb.predict(X_val_scaled)


In [5]:
# Calculate accuracy
accuracy = accuracy_score(y_val_encoded, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

# Print classification report
print("Classification Report:")
print(classification_report(y_val_encoded, y_pred, target_names=le.classes_))

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': xgb.feature_importances_
})
feature_importance = feature_importance.sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

Accuracy: 62.12%
Classification Report:
                   precision    recall  f1-score   support

   diagnosis_adrd       0.47      0.42      0.44        96
diagnosis_control       0.71      0.78      0.74       197
    diagnosis_mci       0.39      0.32      0.35        37

         accuracy                           0.62       330
        macro avg       0.52      0.51      0.51       330
     weighted avg       0.61      0.62      0.61       330


Top 10 Most Important Features:
                             feature  importance
76         alphaRatioUV_sma3nz_amean    0.054748
16     loudness_sma3_meanRisingSlope    0.046494
78         slopeUV0-500_sma3nz_amean    0.041440
77    hammarbergIndexUV_sma3nz_amean    0.025623
58          alphaRatioV_sma3nz_amean    0.021265
56  F3amplitudeLogRelF0_sma3nz_amean    0.019922
62          slopeV0-500_sma3nz_amean    0.019149
55     F3bandwidth_sma3nz_stddevNorm    0.018845
85         MeanUnvoicedSegmentLength    0.018568
10               loud

In [6]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
import numpy as np

# Define parameter space for RandomizedSearchCV
random_params = {
    'n_estimators': randint(10, 1500),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 7),
    'gamma': uniform(0, 0.5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1)
}

# Initialize base model
xgb = XGBClassifier(
    objective='multi:softprob',
    random_state=42
)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=random_params,
    n_iter=200,
    scoring='accuracy',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit and get best parameters
random_search.fit(X_train_scaled, y_train_encoded)

# Get best parameters and score
print("Best parameters:", random_search.best_params_)
print("Best score:", random_search.best_score_)

# Train final model with best parameters
best_model = XGBClassifier(**random_search.best_params_)
best_model.fit(X_train_scaled, y_train_encoded)

# Evaluate on val set
val_score = best_model.score(X_val_scaled, y_val_encoded)
print("Val score:", val_score)

y_pred = best_model.predict(X_val_scaled)
accuracy = accuracy_score(y_val_encoded, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

Fitting 5 folds for each of 200 candidates, totalling 1000 fits
[CV 3/5] END colsample_bytree=0.749816047538945, gamma=0.4753571532049581, learning_rate=0.22959818254342154, max_depth=7, min_child_weight=5, n_estimators=131, reg_alpha=0.15599452033620265, reg_lambda=0.05808361216819946, subsample=0.9464704583099741;, score=0.536 total time=   1.8s
[CV 1/5] END colsample_bytree=0.7216968971838151, gamma=0.2623782158161189, learning_rate=0.13958350559263474, max_depth=3, min_child_weight=3, n_estimators=1092, reg_alpha=0.3998609717152555, reg_lambda=0.04666566321361543, subsample=0.9895022075365837;, score=0.519 total time=   3.7s
[CV 2/5] END colsample_bytree=0.7216968971838151, gamma=0.2623782158161189, learning_rate=0.13958350559263474, max_depth=3, min_child_weight=3, n_estimators=1092, reg_alpha=0.3998609717152555, reg_lambda=0.04666566321361543, subsample=0.9895022075365837;, score=0.574 total time=   3.6s
[CV 3/5] END colsample_bytree=0.7216968971838151, gamma=0.2623782158161189, 

In [8]:
y_pred = best_model.predict(X_val_scaled)
accuracy = accuracy_score(y_val_encoded, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

Accuracy: 63.33%


In [7]:
# Test set preparation
test_features = pd.read_csv('test_features.csv')
aggregated_test_features = test_features.groupby('uid')[feature_columns].mean().reset_index()
X_test_scaled = scaler.transform(aggregated_test_features[feature_columns])

# Prediction probabilities
y_pred_probs = best_model.predict_proba(X_test_scaled)
print("Prediction Probabilities:")
print(y_pred_probs)

Prediction Probabilities:
[[1.2480108e-01 8.7490302e-01 2.9584969e-04]
 [1.3719773e-01 8.6011332e-01 2.6889744e-03]
 [7.3870309e-02 8.3785236e-01 8.8277370e-02]
 ...
 [1.0214263e-01 6.7567360e-01 2.2218375e-01]
 [4.4919601e-01 4.9491709e-01 5.5886947e-02]
 [4.6561489e-01 3.7202564e-01 1.6235951e-01]]
